# DuckDB Benchmark – TPC-H 100 GB

In [ ]:
import duckdb
import os
import sys
import psutil
import time

PARQUET_DIR = "/net/pr2/projects/plgrid/plggmpr2025/parquet"
RESULTS_DIR = "/net/pr2/projects/plgrid/plggmpr2025/results"
os.makedirs(RESULTS_DIR, exist_ok=True)

total_ram_gb = psutil.virtual_memory().total / 1e9
available_ram_gb = psutil.virtual_memory().available / 1e9
print(f"Łączna RAM: {total_ram_gb:.0f} GB")
print(f"Dostępna RAM: {available_ram_gb:.0f} GB")

con = duckdb.connect(database=':memory:', config={
    'memory_limit': '128GB',
    'threads': os.cpu_count(),
})

print(f"\nDuckDB skonfigurowany:")
print(f"  memory_limit: 128GB")
print(f"  threads: {os.cpu_count()}")

In [ ]:
tables = ['lineitem', 'orders', 'customer', 'part', 'supplier', 'partsupp', 'nation', 'region']

for table in tables:
    path = os.path.join(PARQUET_DIR, f"{table}.parquet")
    con.execute(f"CREATE VIEW {table} AS SELECT * FROM parquet_scan('{path}')")
    count = con.execute(f"SELECT COUNT(*) FROM {table}").fetchone()[0]
    print(f"  {table}: {count:,} wierszy")

In [ ]:
QUERIES = {
    "Q1_pricing_summary": """
        SELECT
            l_returnflag,
            l_linestatus,
            SUM(l_quantity) AS sum_qty,
            SUM(l_extendedprice) AS sum_base_price,
            SUM(l_extendedprice * (1 - l_discount)) AS sum_disc_price,
            SUM(l_extendedprice * (1 - l_discount) * (1 + l_tax)) AS sum_charge,
            AVG(l_quantity) AS avg_qty,
            AVG(l_extendedprice) AS avg_price,
            AVG(l_discount) AS avg_disc,
            COUNT(*) AS count_order
        FROM lineitem
        WHERE l_shipdate <= DATE '1998-12-01' - INTERVAL '90' DAY
        GROUP BY l_returnflag, l_linestatus
        ORDER BY l_returnflag, l_linestatus
    """,

    "Q3_shipping_priority": """
        SELECT
            l_orderkey,
            SUM(l_extendedprice * (1 - l_discount)) AS revenue,
            o_orderdate,
            o_shippriority
        FROM customer, orders, lineitem
        WHERE c_mktsegment = 'BUILDING'
          AND c_custkey = o_custkey
          AND l_orderkey = o_orderkey
          AND o_orderdate < DATE '1995-03-15'
          AND l_shipdate > DATE '1995-03-15'
        GROUP BY l_orderkey, o_orderdate, o_shippriority
        ORDER BY revenue DESC, o_orderdate
        LIMIT 10
    """,

    "Q5_local_supplier_volume": """
        SELECT
            n_name,
            SUM(l_extendedprice * (1 - l_discount)) AS revenue
        FROM customer, orders, lineitem, supplier, nation, region
        WHERE c_custkey = o_custkey
          AND l_orderkey = o_orderkey
          AND l_suppkey = s_suppkey
          AND c_nationkey = s_nationkey
          AND s_nationkey = n_nationkey
          AND n_regionkey = r_regionkey
          AND r_name = 'ASIA'
          AND o_orderdate >= DATE '1994-01-01'
          AND o_orderdate < DATE '1994-01-01' + INTERVAL '1' YEAR
        GROUP BY n_name
        ORDER BY revenue DESC
    """,

    "Q6_revenue_change": """
        SELECT
            SUM(l_extendedprice * l_discount) AS revenue
        FROM lineitem
        WHERE l_shipdate >= DATE '1994-01-01'
          AND l_shipdate < DATE '1994-01-01' + INTERVAL '1' YEAR
          AND l_discount BETWEEN 0.06 - 0.01 AND 0.06 + 0.01
          AND l_quantity < 24
    """,

    "Q10_returned_item_reporting": """
        SELECT
            c_custkey,
            c_name,
            SUM(l_extendedprice * (1 - l_discount)) AS revenue,
            c_acctbal,
            n_name,
            c_address,
            c_phone,
            c_comment
        FROM customer, orders, lineitem, nation
        WHERE c_custkey = o_custkey
          AND l_orderkey = o_orderkey
          AND o_orderdate >= DATE '1993-10-01'
          AND o_orderdate < DATE '1993-10-01' + INTERVAL '3' MONTH
          AND l_returnflag = 'R'
          AND c_nationkey = n_nationkey
        GROUP BY c_custkey, c_name, c_acctbal, c_phone, n_name, c_address, c_comment
        ORDER BY revenue DESC
        LIMIT 20
    """,

    "Q21_suppliers_waiting": """
        SELECT
            s_name,
            COUNT(*) AS numwait
        FROM supplier, lineitem l1, orders, nation
        WHERE s_suppkey = l1.l_suppkey
          AND o_orderkey = l1.l_orderkey
          AND o_orderstatus = 'F'
          AND l1.l_receiptdate > l1.l_commitdate
          AND EXISTS (
              SELECT * FROM lineitem l2
              WHERE l2.l_orderkey = l1.l_orderkey
                AND l2.l_suppkey <> l1.l_suppkey
          )
          AND NOT EXISTS (
              SELECT * FROM lineitem l3
              WHERE l3.l_orderkey = l1.l_orderkey
                AND l3.l_suppkey <> l1.l_suppkey
                AND l3.l_receiptdate > l3.l_commitdate
          )
          AND s_nationkey = n_nationkey
          AND n_name = 'SAUDI ARABIA'
        GROUP BY s_name
        ORDER BY numwait DESC, s_name
        LIMIT 100
    """,
}

In [ ]:
sys.path.insert(0, "/net/pr2/projects/plgrid/plggmpr2025/scripts")
from benchmark_utils import BenchmarkRunner

runner = BenchmarkRunner("DuckDB", RESULTS_DIR)

print("=== DuckDB Benchmark ===")
for qname, sql in QUERIES.items():
    runner.run(
        query_name=qname,
        func=lambda q=sql: con.execute(q).fetchall(),
        n_runs=3
    )

runner.save_results("duckdb_results.json")

In [ ]:
import pandas as pd

df = pd.DataFrame(runner.results)
summary = df.groupby('query')['elapsed_s'].agg(['min', 'mean', 'max', 'std']).round(2)
print("\n=== Podsumowanie DuckDB ===")
print(summary.to_string())